# SparseSAM — quick-start demo + profiling

This notebook shows how to:

1. Patch a stock SAM-HQ encoder with **SparseSAM** via the unified `algos.registry`.
2. Compare baseline vs SparseSAM on a single image (mask + encoder latency + GPU memory).
3. Sweep across density ratios to chart the speed/quality trade-off.
4. Profile per-block (attention vs MLP, windowed vs global) before/after patching.

Run from the repo root:
```bash
jupyter lab notebooks/sparsesam_demo.ipynb
```

**Requirements:** `./ckts/sam_hq_vit_l.pth`, an A100/RTX 3090 (sm80+), the `algos/` package importable from CWD.

## 0. Setup

In [ ]:
import os, sys, time, gc
from pathlib import Path

# Make repo root + sam-hq importable.
_REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_REPO))
sys.path.insert(0, str(_REPO / 'algos' / '3rd_party' / 'sam-hq'))
os.chdir(_REPO)
print('Working dir:', os.getcwd())

import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt

from segment_anything import sam_model_registry, SamPredictor
from algos.registry import apply_sam, remove_all_sam, sam_algo_choices, REGISTRATION_ERRORS

print('Registered SAM algos:', sam_algo_choices())
if REGISTRATION_ERRORS:
    print('Skipped (missing deps):', list(REGISTRATION_ERRORS.keys()))
print('GPU:', torch.cuda.get_device_name(0), 'sm', torch.cuda.get_device_capability(0))

In [ ]:
# Load the dense SAM-HQ ViT-L baseline once. We'll patch/unpatch in place.
DEVICE = 'cuda'
CKPT   = './ckts/sam_hq_vit_l.pth'
MODEL  = 'vit_l'

sam = sam_model_registry[MODEL](checkpoint=CKPT).to(DEVICE).eval()
predictor = SamPredictor(sam)
print(f'Loaded SAM-HQ {MODEL} from {CKPT}')
print(f'Encoder blocks: {len(sam.image_encoder.blocks)}  '
      f'(global={sum(1 for b in sam.image_encoder.blocks if b.window_size == 0)})')

## 1. Helpers — timing, memory, and plotting

In [ ]:
def reset_gpu_stats():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


@torch.no_grad()
def time_encoder(predictor, img, n_warmup=2, n_runs=5):
    """Encoder-only wall-clock + peak memory after `set_image`."""
    for _ in range(n_warmup):
        predictor.reset_image(); predictor.set_image(img)
    torch.cuda.synchronize()
    reset_gpu_stats()
    times = []
    for _ in range(n_runs):
        predictor.reset_image()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        predictor.set_image(img)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    mem_mb = torch.cuda.max_memory_allocated() / 1024**2
    return float(np.mean(times) * 1000), float(np.std(times) * 1000), mem_mb


@torch.no_grad()
def predict_box(predictor, box_xyxy, hq=False):
    masks, ious, _ = predictor.predict(
        box=np.array(box_xyxy), multimask_output=False, hq_token_only=hq,
    )
    return masks[0], float(ious[0])


def show_with_mask(ax, img, mask, title, box=None, color='#0891b2', alpha=0.55):
    ax.imshow(img)
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    rgba = np.array([int(color[i:i+2], 16)/255 for i in (1, 3, 5)] + [alpha])
    overlay[mask] = rgba
    ax.imshow(overlay)
    if box is not None:
        x0, y0, x1, y1 = box
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                   fill=False, ec='#facc15', lw=2))
    ax.set_title(title, fontsize=11); ax.set_xticks([]); ax.set_yticks([])

## 2. Load a demo image + prompt

Using the butterfly image from `input_imgs/` with a box prompt. Same image
as Figure 1 of the paper.

In [ ]:
img_bgr = cv2.imread('./input_imgs/example1.png')
img     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
BOX     = [306, 132, 925, 893]   # butterfly bbox

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img)
ax.add_patch(plt.Rectangle((BOX[0], BOX[1]), BOX[2]-BOX[0], BOX[3]-BOX[1],
                            fill=False, ec='#facc15', lw=2))
ax.set_title(f'input  {img.shape[1]}×{img.shape[0]}'); ax.axis('off')
plt.show()

## 3. Baseline run

Stock dense SAM-HQ. No patch applied — `algos.registry.apply_sam(...)`
with `name='none'` would be a no-op, but we just skip it entirely.

In [ ]:
remove_all_sam(sam.image_encoder)   # idempotent; ensure clean state

mean_ms, std_ms, mem_mb = time_encoder(predictor, img)
mask_base, iou_base     = predict_box(predictor, BOX, hq=True)

print(f'baseline  enc/img = {mean_ms:.1f} ± {std_ms:.1f} ms   '
      f'peak GPU = {mem_mb:.0f} MB   pred-IoU = {iou_base:.3f}')

baseline_ms = mean_ms
baseline_mb = mem_mb

## 4. Patch with SparseSAM and re-measure

Three lines: import → `apply_sam` → run. Reverting is one line: `remove_all_sam`.

In [ ]:
RATIO = 0.5   # keep 50% of tokens

apply_sam(sam.image_encoder, name='sparsesam', ratio=RATIO, mlp_merge=True)

mean_ms, std_ms, mem_mb = time_encoder(predictor, img)
mask_sparse, iou_sparse = predict_box(predictor, BOX, hq=True)

print(f'sparsesam r={RATIO}   enc/img = {mean_ms:.1f} ± {std_ms:.1f} ms   '
      f'peak GPU = {mem_mb:.0f} MB   pred-IoU = {iou_sparse:.3f}')
print(f'  speedup vs baseline: ×{baseline_ms / mean_ms:.2f}  '
      f'memory drop: {100 * (1 - mem_mb / baseline_mb):.0f}%')

remove_all_sam(sam.image_encoder)   # revert

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
show_with_mask(axes[0], img, mask_base,   f'baseline  ({baseline_ms:.0f} ms)', box=BOX)
show_with_mask(axes[1], img, mask_sparse, f'SparseSAM r={RATIO}  (×{baseline_ms/mean_ms:.2f})', box=BOX)
plt.tight_layout(); plt.show()

## 5. Sweep across density ratios

Drop into the full registry to compare SparseSAM at multiple densities against
the dense baseline.

In [ ]:
RATIOS = [0.3, 0.5, 0.7]

rows = []
remove_all_sam(sam.image_encoder)
ms, _, mb = time_encoder(predictor, img)
rows.append(('none', 1.00, ms, mb, predict_box(predictor, BOX, hq=True)[1]))

for r in RATIOS:
    remove_all_sam(sam.image_encoder)
    apply_sam(sam.image_encoder, name='sparsesam', ratio=r, mlp_merge=True)
    ms, _, mb = time_encoder(predictor, img)
    iou = predict_box(predictor, BOX, hq=True)[1]
    rows.append(('sparsesam', r, ms, mb, iou))

remove_all_sam(sam.image_encoder)

import pandas as pd
df = pd.DataFrame(rows, columns=['algo', 'ratio', 'enc_ms', 'peak_mb', 'pred_iou'])
df['speedup'] = df['enc_ms'].iloc[0] / df['enc_ms']
df['mem_drop'] = 1 - df['peak_mb'] / df['peak_mb'].iloc[0]
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax1, ax2 = axes
ax1.plot(df['ratio'], df['speedup'], 'o-', color='#0891b2', label='encoder speedup')
ax1.axhline(1, color='gray', linestyle='--', linewidth=0.6)
ax1.set_xlabel('density (fraction of tokens kept)')
ax1.set_ylabel('encoder speedup vs baseline')
ax1.set_title('Speed')
ax1.set_xlim(0.2, 1.05); ax1.grid(alpha=0.3)

ax2.plot(df['ratio'], df['pred_iou'], 'o-', color='#0891b2', label='pred-IoU')
ax2.axhline(df['pred_iou'].iloc[0], color='gray', linestyle='--', linewidth=0.6,
            label=f'baseline ({df["pred_iou"].iloc[0]:.3f})')
ax2.set_xlabel('density'); ax2.set_ylabel('predicted IoU (single image)')
ax2.set_title('Quality')
ax2.set_xlim(0.2, 1.05); ax2.grid(alpha=0.3); ax2.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 6. Side-by-side masks across ratios

In [ ]:
remove_all_sam(sam.image_encoder)
mask_base, _ = predict_box(predictor, BOX, hq=True)

masks = [('baseline', None, mask_base)]
for r in RATIOS:
    remove_all_sam(sam.image_encoder)
    apply_sam(sam.image_encoder, name='sparsesam', ratio=r, mlp_merge=True)
    _, _, _ = time_encoder(predictor, img, n_warmup=0, n_runs=1)   # populate features
    masks.append(('sparsesam', r, predict_box(predictor, BOX, hq=True)[0]))
remove_all_sam(sam.image_encoder)

fig, axes = plt.subplots(1, len(masks), figsize=(4 * len(masks), 4.5))
for ax, (algo, r, m) in zip(axes, masks):
    title = algo if r is None else f'{algo}  r={r}'
    show_with_mask(ax, img, m, title, box=BOX)
plt.tight_layout(); plt.show()